In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [6]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
logits

tensor([[-0.0056, -0.0482,  0.0243,  0.0105,  0.0739, -0.0525, -0.0151, -0.0111,
         -0.0169, -0.0149]], device='mps:0', grad_fn=<LinearBackward0>)

In [9]:
pred_proba = nn.Softmax(dim=1)(logits)
y_pred = pred_proba.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([4], device='mps:0')


In [12]:
input_image = torch.rand(3,28,28)
print(input_image.size())
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 28, 28])
torch.Size([3, 784])


In [13]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [14]:
print(f"Before ReLU: {hidden1}\n")
hidden1 = nn.ReLU()(hidden1)
print(f"after ReLU: {hidden1}")

Before ReLU: tensor([[-0.4055,  0.1356,  0.6246, -0.1029,  0.2555, -0.0255, -0.3870,  0.2101,
          0.2223,  0.0110, -0.2706,  0.2697, -0.2267, -0.6197, -0.1644, -0.0090,
          0.2233, -0.2502, -0.3549, -0.4457],
        [-0.1133, -0.2265,  0.2470, -0.1409,  0.6322, -0.2891, -0.3616, -0.0606,
         -0.2345, -0.0057, -0.0897,  0.4727,  0.1572, -0.3382, -0.4090, -0.1904,
          0.2883, -0.1035,  0.2355, -0.3691],
        [-0.5018, -0.2991,  0.1705, -0.2677,  0.5976, -0.2548, -0.4106,  0.2611,
          0.2722,  0.1660, -0.3119,  0.4992, -0.2205, -0.2722, -0.2712,  0.1324,
          0.5597, -0.3086, -0.3185, -0.3267]], grad_fn=<AddmmBackward0>)

after ReLU: tensor([[0.0000, 0.1356, 0.6246, 0.0000, 0.2555, 0.0000, 0.0000, 0.2101, 0.2223,
         0.0110, 0.0000, 0.2697, 0.0000, 0.0000, 0.0000, 0.0000, 0.2233, 0.0000,
         0.0000, 0.0000],
        [0.0000, 0.0000, 0.2470, 0.0000, 0.6322, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.4727, 0.1572, 0.0000, 0.000

In [16]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)
softmax = nn.Softmax(dim=1)
pred_proba = softmax(logits)
pred_proba

tensor([[0.0911, 0.1363, 0.0969, 0.0950, 0.1198, 0.0870, 0.0993, 0.0822, 0.0955,
         0.0969],
        [0.0871, 0.1422, 0.1059, 0.0978, 0.1055, 0.0767, 0.1008, 0.0924, 0.0935,
         0.0981],
        [0.0843, 0.1314, 0.1123, 0.0874, 0.1177, 0.0818, 0.1180, 0.0829, 0.0850,
         0.0993]], grad_fn=<SoftmaxBackward0>)

In [17]:
print(f"Model structure: {model}\n\n")
for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0349,  0.0094, -0.0283,  ...,  0.0256, -0.0125, -0.0137],
        [-0.0266,  0.0320, -0.0272,  ...,  0.0107,  0.0084, -0.0113]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0122, 0.0318], device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0361,  0.0127,  0.0142,  ...,  0.0062, -0.0006, -0.0164],
        [ 0.0025, -0.0372,  0.0209,  ...,  0.0356, -0.0172, -0.0135]],
       device='mps:0', grad_fn=<SliceBa